In [ ]:
import os
from pathlib import Path
import re
import warnings
import threading
import pandas as pd
import numpy as np
import scipy.sparse as sp
from joblib import Parallel, delayed
import joblib
import json

import xgboost as xgb
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings('ignore')

# Locate the repository data; AV_DATA_DIR can override this on hosted runtimes.
candidates = [Path.cwd(), *Path.cwd().parents]
default_root = next((path for path in candidates if (path / 'data').is_dir()), Path.cwd())
REPO_ROOT = Path(os.getenv('AV_PROJECT_ROOT', default_root)).resolve()
DATA_DIR = Path(os.getenv('AV_DATA_DIR', REPO_ROOT / 'data'))

def load_data(directory):
    train_path = os.path.join(directory, "train.csv")
    dev_path = os.path.join(directory, "dev.csv")
    
    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Path not found: {train_path}")
        
    train = pd.read_csv(train_path)
    dev = pd.read_csv(dev_path)
    
    print(f"✅ Data Loaded | Train: {len(train):,} | Dev: {len(dev):,}")
    return train, dev

train_df, dev_df = load_data(DATA_DIR)
train_df.head()

# Verify dataset shape, class distribution, and missing values
print(train_df.shape, dev_df.shape)
print(train_df['label'].value_counts())
print(train_df.isnull().sum())

In [ ]:
# 0. Text regularization: remove emails, URLs, dates, phone numbers, and repeated characters
def normalize_text(text):
    text = str(text)
    text = re.sub(r'[\w.+-]+@[\w.]+\.[a-z]{2,}', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', ' ', text)
    text = re.sub(r'\d{1,2}:\d{2}(:\d{2})?', ' ', text)
    text = re.sub(r'\(?\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}', ' ', text)
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    text = re.sub(r'([!?.]){2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 1. Stylometric feature extraction: 24 hand-crafted features per text (word/sentence stats, punctuation ratios, suffix patterns)
def process_text_stylometric(text):
    text = str(text)
    words = text.split()
    ln = len(words)

    if ln < 5:
        return np.zeros(24, dtype='float32')

    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    word_lens = [len(w) for w in words]
    sent_lens = [len(s.split()) for s in sentences] if sentences else [ln]

    return np.array([
        ln,
        np.mean(word_lens),
        np.mean(sent_lens),
        len(set(words)) / ln, # Type-Token Ratio
        
        np.std(word_lens),
        np.std(sent_lens),
        np.max(word_lens),
        np.median(sent_lens),

        text.count(',') / ln,
        text.count('.') / ln,
        text.count('!') / ln,
        text.count('?') / ln,
        text.count(';') / ln,
        text.count(':') / ln,
        text.count('"') / ln,
        text.count("'") / ln,

        sum(w.isupper() for w in words) / ln,
        sum(len(w) > 6 for w in words) / ln,
        sum(len(w) <= 3 for w in words) / ln,
        sum(w[0].isupper() for w in words if w) / ln,

        sum(w.endswith('ly') for w in words) / ln,
        sum(w.endswith(('tion', 'sion')) for w in words) / ln,
        sum(w.endswith('ing') for w in words) / ln,
        sum(w.endswith('ed') for w in words) / ln,
    ], dtype='float32')

# Extract stylometric features for both texts in parallel, then compute pairwise interactions (diff, ratio, mean)
def fast_feature_extraction(df):
    texts_1 = df['text_1'].apply(normalize_text).values
    texts_2 = df['text_2'].apply(normalize_text).values
    all_texts = np.concatenate([texts_1, texts_2])
    mid = len(df)

    style_results = Parallel(n_jobs=-1, batch_size=1000)(
        delayed(process_text_stylometric)(t) for t in all_texts
    )
    s1 = np.array(style_results[:mid])
    s2 = np.array(style_results[mid:])

    diff  = np.abs(s1 - s2)
    ratio = np.minimum(s1, s2) / (np.maximum(s1, s2) + 1e-9)
    mean  = (s1 + s2) / 2

    combined = np.hstack([diff, ratio, mean]).astype('float32')
    return sp.csr_matrix(combined, dtype='float32'), texts_1, texts_2

# 2. TF-IDF Fitting: word-level (1-2 gram), character-level (2-4 gram), and function word vocabularies
FUNCTION_WORDS = [
    'the', 'a', 'an', 'and', 'or', 'but', 'if', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should',
    'may', 'might', 'shall', 'that', 'which', 'who', 'whom', 'this', 'these', 'those',
    'i', 'we', 'you', 'he', 'she', 'they', 'it', 'my', 'your', 'his', 'her',
    'however', 'therefore', 'moreover', 'furthermore', 'although', 'though',
    'because', 'since', 'while', 'when', 'where', 'what', 'how', 'why', 'just', 'also',
    'very', 'really', 'quite', 'rather', 'so', 'too', 'even', 'still', 'already',
    'not', 'only', 'then', 'here', 'there', 'now', 'never', 'always', 'well',
    'me', 'us', 'him', 'them', 'its', 'our', 'their', 'about', 'into', 'through',
    'over', 'under', 'before', 'after', 'cannot', 'without',

    'of the', 'in the', 'to the', 'on the', 'for the', 'at the', 'by the',
    'with the', 'from the', 'and the', 'of a', 'in a', 'to a', 'for a', 'with a',

    'it is', 'it was', 'there is', 'there are', 'there was', 'there were',
    'to be', 'will be', 'would be', 'could be', 'should be', 'may be',
    'is a', 'is the', 'was a', 'was the', 'are the', 'were the',

    'do not', 'does not', 'did not', 'will not', 'would not',
    'could not', 'should not', 'have not', 'has not', 'had not',
    'is not', 'are not', 'was not', 'were not',

    'i am', 'i was', 'i have', 'i had', 'i will', 'i would', 'i think',
    'i know', 'i do', 'i can', 'i could', 'i feel', 'i want', 'i need',
    'we are', 'we have', 'we will', 'we were', 'we had',
    'you are', 'you have', 'you will', 'you were', 'you can',
    'he was', 'he is', 'he had', 'he would', 'he said',
    'she was', 'she is', 'she had', 'she would', 'she said',
    'they are', 'they were', 'they have', 'they had', 'they will',

    'to me', 'for me', 'of it', 'and i', 'but i', 'so i', 'that i',

    'as well', 'as a', 'such as', 'as the', 'up to', 'out of',
    'in order', 'in fact', 'in addition', 'at least', 'at all',
    'of course', 'as if', 'even if', 'even though',
    'so that', 'in that', 'now that', 'given that',
    'due to', 'based on', 'rather than'
]

print("Step 1: Normalizing texts...")
train_df['text_1_clean'] = train_df['text_1'].apply(normalize_text)
train_df['text_2_clean'] = train_df['text_2'].apply(normalize_text)
dev_df['text_1_clean']   = dev_df['text_1'].apply(normalize_text)
dev_df['text_2_clean']   = dev_df['text_2'].apply(normalize_text)

print("Step 2: TF-IDF Fitting...")
all_txt = pd.concat([
    train_df['text_1_clean'], train_df['text_2_clean'],
    dev_df['text_1_clean'],   dev_df['text_2_clean']
])

v_w  = TfidfVectorizer(ngram_range=(1, 2), max_features=7000, dtype=np.float32)
v_c  = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=7000, dtype=np.float32)
v_fw = TfidfVectorizer(vocabulary=FUNCTION_WORDS, ngram_range=(1, 2), dtype=np.float32)

v_w.fit(all_txt)
v_c.fit(all_txt)
v_fw.fit(all_txt)

print("Step 3: Style Feature Extraction...")
train_s, _, _  = fast_feature_extraction(train_df)
dev_s,   _, _  = fast_feature_extraction(dev_df)


# 3. Feature matrix: concatenate TF-IDF diff/product and cosine similarities for each text pair
def build_matrix(df, s):
    w1  = v_w.transform(df['text_1_clean'])
    c1  = v_c.transform(df['text_1_clean'])
    fw1 = v_fw.transform(df['text_1_clean'])

    w2  = v_w.transform(df['text_2_clean'])
    c2  = v_c.transform(df['text_2_clean'])
    fw2 = v_fw.transform(df['text_2_clean'])

    x1 = sp.hstack([w1, c1])
    x2 = sp.hstack([w2, c2])

    diff    = abs(x1 - x2)
    prod    = x1.multiply(x2)

    fw_diff = abs(fw1 - fw2)
    fw_prod = fw1.multiply(fw2)

    cos    = paired_cosine_distances(x1,  x2).reshape(-1, 1).astype('float32')
    fw_cos = paired_cosine_distances(fw1, fw2).reshape(-1, 1).astype('float32')

    return sp.hstack(
        [diff, prod, fw_diff, fw_prod, s, cos, fw_cos],
        format='csr', dtype='float32'
    )

print("Step 4: Matrix Consolidation...")
train_features = build_matrix(train_df, train_s)
dev_features   = build_matrix(dev_df,   dev_s)
print(f"Completed! Shape: {train_features.shape}")

In [ ]:
results = {}
models = {}

# Train XGBoost on GPU 0 with early stopping; store dev probabilities
def train_xgb():
    print("🚀 XGBoost: GPU 0 Start")
    model = xgb.XGBClassifier(
        n_estimators=7000,
        learning_rate=0.01,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        tree_method='hist',
        device='cuda:0',
        max_bin=128,
        random_state=10879360,
        early_stopping_rounds=200,
        eval_metric='logloss'
    )
    model.fit(
        train_features, train_df['label'],
        eval_set=[(dev_features, dev_df['label'])],
        verbose=100
    )
    results['xgb'] = model.predict_proba(dev_features)[:, 1]
    models['xgb'] = model
    print("✅ XGBoost: Done")

# Train LightGBM on GPU 1 with early stopping; store dev probabilities
def train_lgb():
    print("🥊 LightGBM: GPU 1 Start")
    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.01,
        num_leaves=127,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=10879360,
        device='gpu',
        gpu_platform_id=0,
        gpu_device_id=1,
        max_bin=127
    )
    model.fit(
        train_features, train_df['label'],
        eval_set=[(dev_features, dev_df['label'])],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=100)
        ]
    )
    results['lgb'] = model.predict_proba(dev_features)[:, 1]
    models['lgb'] = model
    print("✅ LightGBM: Done")

# Run both models in parallel using threading to utilize dual GPUs simultaneously
t1 = threading.Thread(target=train_xgb)
t2 = threading.Thread(target=train_lgb)

t1.start(); t2.start()
t1.join(); t2.join()

probs_xgb = results['xgb']
probs_lgb = results['lgb']

print("\n🏁 Dual-GPU Training Complete.")

In [ ]:
# Feature Importance Analysis: map feature indices to human-readable names grouped by type
style_cols = [
    'ln', 'mean_word_len', 'mean_sent_len', 'ttr',
    'std_word_len', 'std_sent_len', 'max_word_len', 'median_sent_len',
    'comma_r', 'period_r', 'exclaim_r', 'question_r',
    'semicolon_r', 'colon_r', 'quote_r', 'apos_r',
    'upper_r', 'long_word_r', 'short_word_r', 'cap_r',
    'adv_r', 'nominal_suffix_r', 'ing_r', 'ed_r'
]

feature_names = (
    [f'diff_w_{v}'   for v in v_w.get_feature_names_out()] +
    [f'diff_c_{v}'   for v in v_c.get_feature_names_out()] +
    [f'prod_w_{v}'   for v in v_w.get_feature_names_out()] +
    [f'prod_c_{v}'   for v in v_c.get_feature_names_out()] +
    [f'fw_diff_{v}'  for v in v_fw.get_feature_names_out()] +
    [f'fw_prod_{v}'  for v in v_fw.get_feature_names_out()] +
    [f'style_diff_{c}'  for c in style_cols] +
    [f'style_ratio_{c}' for c in style_cols] +
    [f'style_mean_{c}'  for c in style_cols] +
    ['cos', 'fw_cos']
)

# Fallback to indices if feature count doesn't match the matrix shape
n_features = train_features.shape[1]
if len(feature_names) != n_features:
    print(f"⚠️ Feature name mismatch ({len(feature_names)} vs {n_features}) — falling back to indices")
    feature_names = [str(i) for i in range(n_features)]

def get_group(name):
    if name.startswith('diff_w'):      return 'word_diff'
    if name.startswith('prod_w'):      return 'word_prod'
    if name.startswith('diff_c'):      return 'char_diff'
    if name.startswith('prod_c'):      return 'char_prod'
    if name.startswith('fw_diff'):     return 'fw_diff'
    if name.startswith('fw_prod'):     return 'fw_prod'
    if name.startswith('style_diff'):  return 'style_diff'
    if name.startswith('style_ratio'): return 'style_ratio'
    if name.startswith('style_mean'):  return 'style_mean'
    if name in ('cos', 'fw_cos'):      return 'cosine'
    return 'other'

# Print group-level and top-20 individual feature importances for each model
for model_name in ['xgb', 'lgb']:
    imp = models[model_name].feature_importances_
    fi = pd.DataFrame({'feature': feature_names, 'importance': imp})

    fi['group'] = fi['feature'].apply(get_group)
    group_sum = (fi.groupby('group')['importance']
                   .sum()
                   .sort_values(ascending=False)
                   .reset_index())
    group_sum['pct'] = (group_sum['importance'] / group_sum['importance'].sum() * 100).round(1)

    print(f"\n{'='*45}")
    print(f"  {model_name.upper()} Feature Importance by Group")
    print(f"{'='*45}")
    print(group_sum.to_string(index=False))

    print(f"\n  {model_name.upper()} Top 20 Individual Features")
    print(f"{'-'*45}")
    print(fi.nlargest(20, 'importance')[['feature', 'importance']].to_string(index=False))

In [ ]:
print("\n--- Individual Model Scores ---")

# Sweep thresholds [0.2, 0.8] to find the optimal decision boundary maximising macro F1
best_f1_xgb, best_th_xgb = 0, 0
for th in np.arange(0.2, 0.8, 0.01):
    preds = (probs_xgb >= th).astype(int)
    score = f1_score(dev_df['label'], preds, average='macro')
    if score > best_f1_xgb:
        best_f1_xgb, best_th_xgb = score, th

print(f"🚀 XGBoost Max F1: {best_f1_xgb:.4f} (Threshold: {best_th_xgb:.2f})")

best_f1_lgb, best_th_lgb = 0, 0
for th in np.arange(0.2, 0.8, 0.01):
    preds = (probs_lgb >= th).astype(int)
    score = f1_score(dev_df['label'], preds, average='macro')
    if score > best_f1_lgb:
        best_f1_lgb, best_th_lgb = score, th

print(f"🥊 LightGBM Max F1: {best_f1_lgb:.4f} (Threshold: {best_th_lgb:.2f})")
print("-" * 35)

In [ ]:
# Grid search over XGB weight [0.2, 0.8] and decision threshold to maximise ensemble macro F1
print("🧪 Searching for the Golden Ratio (XGB vs LGBM)...")

best_overall_f1 = 0
best_weight = 0
best_final_thresh = 0

for w in np.arange(0.2, 0.81, 0.01):
    curr_probs = (probs_xgb * w) + (probs_lgb * (1 - w))
    
    for thresh in np.arange(0.2, 0.8, 0.01):
        preds = (curr_probs >= thresh).astype(int)
        score = f1_score(dev_df['label'], preds, average='macro')
        
        if score > best_overall_f1:
            best_overall_f1 = score
            best_weight = w
            best_final_thresh = thresh

print("\n" + "="*40)
print(f"🎊 SEARCH COMPLETED! 🎊")
print(f"🥇 Best XGB Weight: {best_weight:.2f}")
print(f"🥈 Best LGBM Weight: {1-best_weight:.2f}")
print(f"🥉 Best Threshold: {best_final_thresh:.2f}")
print(f"🚀 MAX F1 SCORE: {best_overall_f1:.4f}")
print("="*40)

# Apply optimal weights and threshold; print final classification report on dev set
final_probs = (probs_xgb * best_weight) + (probs_lgb * (1 - best_weight))
final_preds = (final_probs >= best_final_thresh).astype(int)
print("\n--- Ultimate Ensemble Classification Report ---")
print(classification_report(dev_df['label'], final_preds))

In [ ]:
OUTPUT_DIR = Path(os.getenv('AV_MODEL_DIR', REPO_ROOT / 'models' / 'stylometric-ensemble'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save models
models['xgb'].save_model(os.path.join(OUTPUT_DIR, "xgb_model.json"))
joblib.dump(models['lgb'], os.path.join(OUTPUT_DIR, "lgb_model.pkl"))

# Save vectorizers
joblib.dump(v_w,  os.path.join(OUTPUT_DIR, "tfidf_word.pkl"))
joblib.dump(v_c,  os.path.join(OUTPUT_DIR, "tfidf_char.pkl"))
joblib.dump(v_fw, os.path.join(OUTPUT_DIR, "tfidf_fw.pkl"))

# Save ensemble config
ensemble_config = {
    "xgb_weight":    round(best_weight, 2),
    "lgb_weight":    round(1 - best_weight, 2),
    "threshold":     round(best_final_thresh, 2),
    "best_f1":       round(best_overall_f1, 4),
}
with open(os.path.join(OUTPUT_DIR, "ensemble_config.json"), "w") as f:
    json.dump(ensemble_config, f, indent=2)

print("✅ All models saved.")
print(f"   📦 XGBoost    → xgb_model.json")
print(f"   📦 LightGBM   → lgb_model.pkl")
print(f"   📦 Vectorizers → tfidf_word/char/fw.pkl")
print(f"   📦 Ensemble   → ensemble_config.json")
print(f"\n   XGB weight : {ensemble_config['xgb_weight']}")
print(f"   LGB weight : {ensemble_config['lgb_weight']}")
print(f"   Threshold  : {ensemble_config['threshold']}")
print(f"   Best F1    : {ensemble_config['best_f1']}")